In [ ]:
%load_ext autoreload
%autoreload 2

# Posture sampling with a reach grid

Revisiting posture sampling with a precomputed 1 cm `ReachGrid` (`reach_sdf.py`) instead of
inverse kinematics. A lookup maps a foot position (shoulder frame) to the nearest grid vertex,
`round((x - lower_left) / h)`, and returns every branch's validity, angles and **stored foot**:
an approximate posture, self-consistent (FK of the angles is the returned foot) but up to
`sqrt(3) h / 2` off the requested foot. Pattern: approximate for the many samples, exact
(IK) only for the few that survive scoring.

Climb robot (4 legs, the config's joint limits), body box as keep-out, flat floor with the
planner's foothold density. No collision / terrain checks (the same cost for every variant).

Fixed body (`body0`); sampling footholds and joint angles:

1. Candidate footholds: reach ball vs. ball + grid lookup vs. exact IK.
2. Single leg: n footholds for one leg, with angles.
3. Full stance: n stances (a foothold and angles for every leg).

Sampling variants (2, 3), each per call, i.e. per body:

- **ball + IK per draw**: draw among the footholds in the reach ball, IK on each draw (the
  planner's pattern); many draws are unreachable.
- **IK all, then draw**: IK on every foothold once, draw among the reachable ones, gather.
- **grid all, then draw**: lookup on every foothold once, draw among the reachable ones, gather
  (angles and stored feet).

In [ ]:
import time

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from controlkit.se3 import SE3

from lab.gait_graph import terrain
from lab.gait_graph.planner.climb_model.config import MjModelCfg
from lab.gait_graph.planner.climb_model.mjmodel import make_robot
from lab.kinematics.reach_sdf import Keepout, branches_fn, make_reach_grid, reach_mask

def timed(f, *args, repeats=5):
    # (compile_s, call_s, out): first call minus a call; mean of `repeats` more calls.
    t = time.perf_counter(); out = jax.block_until_ready(f(*args)); first = time.perf_counter() - t
    t = time.perf_counter()
    for _ in range(repeats):
        out = jax.block_until_ready(f(*args))
    call = (time.perf_counter() - t) / repeats
    return max(first - call, 0.0), call, out

In [ ]:
cfg = MjModelCfg()
robot = make_robot(cfg)               # 4 legs, joint limits from the config
leg, L = robot.leg, cfg.num_legs
R = float(jnp.sum(leg.lengths))       # reach ball radius
FOOT_R = cfg.pivot_height + cfg.pad_thickness   # the FK foot (ankle pivot) above the pad face
H = 0.01

body_center, body_half = robot.mount_box(half_height=cfg.body_half_height)
keepouts = [Keepout(robot.mounts[i].inverse(), body_center, body_half) for i in range(L)]

t0 = time.perf_counter()
grid = make_reach_grid(leg, H, keepout=keepouts[0])
print(f"grid {grid.shape}, h {grid.h * 1e3:.1f} mm, built in {time.perf_counter() - t0:.2f}s, "
      f"{(grid.ok.nbytes + grid.thetas.nbytes + grid.feet.nbytes) / 1e6:.0f} MB")
print("limits (deg):", np.rad2deg(np.asarray(leg.limits)).round().tolist())

# One grid for all legs: the body box looks the same from every mount (square, mounts 90 deg
# apart). Up to ties: in each shoulder frame the box is rotated 45 deg, its faces run through
# lattice points diagonally, and float32 decides their side (a few dozen voxels).
masks = [reach_mask(leg, H, keepout=k)[0] for k in keepouts]
print("voxels differing from leg 0's reach set:", [int((masks[0] ^ m).sum()) for m in masks[1:]],
      f"of {int(masks[0].sum()):,}")
g = grid.device()

In [ ]:
# Floor (top face at z = 0) with the planner's foothold density; nominal body at stand height.
scene = terrain.make_scene((((0.0, 0.0, -0.05), (2.0, 1.0, 0.05)),))
fh = terrain.sample_footholds(jax.random.PRNGKey(0), scene, 40_000, edge_margin=0.02)
targets = fh.position + FOOT_R * fh.normal        # where the FK foot must be, per foothold
body0 = SE3(jnp.asarray([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, cfg.stand_height]))
M = targets.shape[0]
print(f"{M:,} footholds")

def to_shoulders(body, X):
    # World points (M, 3) -> each leg's shoulder frame (L, M, 3).
    sh = robot.shoulders(body)
    return jax.vmap(lambda i: jax.vmap(sh[i].inverse().apply)(X))(jnp.arange(L))

## Findings (2026-10-09)

1 cm grid (59 x 71 x 73 cropped; validity, angles, feet for 4 branches: 31 MB, 0.7 s to
build). ~21k footholds on the floor, fixed body, laptop CPU. Times per sample after
compilation; "valid": share of draws with a reachable foothold (every leg, for a stance).

| n | variant | single leg: us / valid / us per valid | full stance: us / valid / us per valid |
|---|---|---|---|
| 16384 | ball + IK per draw | 0.26 / 0.39 / 0.67 | 0.95 / 0.02 / 40.7 |
| 16384 | IK all, then draw | 0.38 / 1.00 / 0.38 | 1.04 / 1.00 / 1.04 |
| 16384 | grid all, then draw | 0.14 / 1.00 / 0.14 | 0.25 / 1.00 / 0.25 |
| 262144 | ball + IK per draw | 0.18 / 0.40 / 0.45 | 0.70 / 0.03 / 27.8 |
| 262144 | IK all, then draw | 0.040 / 1.00 / 0.040 | 0.12 / 1.00 / 0.12 |
| 262144 | grid all, then draw | 0.024 / 1.00 / 0.024 | 0.073 / 1.00 / 0.073 |

- Every call first scans all footholds once (grid ~1.5 ms per call, IK ~4-15 ms); for small n
  that scan is the cost (n = 1: milliseconds per sample), for large n the per-draw gather.
- **ball + IK per draw** wastes draws: 40% of the ball is reachable per leg, so only ~3% of
  4-leg stances are valid: ~28-40 us per valid stance, ~400x the grid.
- **grid vs. IK all**: the same pattern (scan, then draw among the reachable); the grid's
  scan is ~3-10x cheaper, so it wins for small and medium n (4x at 16k for stances), and
  ~1.6x for large n where the gather dominates.
- Full stance vs. single leg (grid): ~2-3x a single leg, not 4x (the per-leg scans are
  batched).
- Grid foot error (stored foot vs. foothold): median 4 mm, max 7 mm.
- Candidates: the ball keeps ~1100 footholds per leg, 60% unreachable; ball + grid keeps
  ~440, matching exact IK up to ~20 boundary footholds each way, at the ball's cost (16 vs.
  15 ns per (leg, foothold); exact IK 175 ns).

## 1. Candidate footholds

Per leg, which footholds the foot can stand on at `body0`. Ball: distance to the shoulder
<= R (the planner's `candidates_at`). Grid: ball, then the lookup. Exact: IK + limits + body
box on every foothold.

In [ ]:
reach_exact = branches_fn(leg, keepout=keepouts[0])

@jax.jit
def cand_ball(body):
    Xs = to_shoulders(body, targets)
    return jnp.sum(Xs ** 2, -1) <= R ** 2

@jax.jit
def cand_grid(body):
    Xs = to_shoulders(body, targets)
    ok = jax.vmap(jax.vmap(lambda x: g.lookup(x)[1].any()))(Xs)
    return ok & (jnp.sum(Xs ** 2, -1) <= R ** 2)

@jax.jit
def cand_ik(body):
    Xs = to_shoulders(body, targets)
    return jax.vmap(jax.vmap(lambda x: reach_exact(x)[0].any()))(Xs)

res = {}
for name, f in (("ball", cand_ball), ("ball + grid", cand_grid), ("exact IK", cand_ik)):
    _, call, out = timed(f, body0)
    res[name] = np.asarray(out)
    print(f"{name:12s} {call * 1e3:7.3f} ms for {L} x {M:,} = {1e9 * call / (L * M):6.2f} ns per (leg, foothold) "
          f"| candidates per leg {res[name].sum(1)}")
ex = res["exact IK"]
for name in ("ball", "ball + grid"):
    r = res[name]
    print(f"{name:12s} vs exact: missed {int((ex & ~r).sum())}, extra {int((r & ~ex).sum())} "
          f"({(r & ~ex).sum() / max(r.sum(), 1):.1%} of its candidates)")

## Samplers

One leg: shoulder-frame footholds `Xs` (M, 3) in, n draws out: foothold index, valid, the
angles of one valid branch (uniform, Gumbel-max) and its foot. Full stance: the same per leg
(vmapped), valid if every leg is.

In [ ]:
from functools import partial

def _draw(key, mask, n):
    # n indices, uniform among mask (with replacement): inverse CDF, O(n log M).
    # (jax.random.categorical with shape=(n,) draws n x M Gumbel noise: 2e10 at 1M x 21k.)
    c = jnp.cumsum(mask)
    u = jax.random.uniform(key, (n,)) * c[-1]
    return jnp.minimum(jnp.searchsorted(c, u, side="right"), mask.shape[0] - 1)

def _branch(key, ok):
    # One valid branch per row, uniformly (Gumbel-max); ok (n, B).
    return jnp.argmax(jnp.where(ok, jax.random.gumbel(key, ok.shape), -jnp.inf), -1)

def leg_ball_ik(key, Xs, n):
    k1, k2 = jax.random.split(key)
    inball = jnp.sum(Xs ** 2, -1) <= R ** 2
    idx = _draw(k1, inball, n)
    ok, th = jax.vmap(reach_exact)(Xs[idx])
    b = _branch(k2, ok)
    return idx, ok.any(-1), th[jnp.arange(n), b], Xs[idx]          # IK: foot exact

def leg_ik_all(key, Xs, n):
    k1, k2 = jax.random.split(key)
    ok, th = jax.vmap(reach_exact)(Xs)                             # every foothold once
    idx = _draw(k1, ok.any(-1), n)
    b = _branch(k2, ok[idx])
    return idx, ok[idx].any(-1), th[idx, b], Xs[idx]

def leg_grid_all(key, Xs, n):
    k1, k2 = jax.random.split(key)
    _, ok, th, feet = jax.vmap(g.lookup)(Xs)                       # every foothold once
    reach = ok.any(-1) & (jnp.sum(Xs ** 2, -1) <= R ** 2)
    idx = _draw(k1, reach, n)
    b = _branch(k2, ok[idx])
    return idx, ok[idx].any(-1), th[idx, b], feet[idx, b]          # stored foot

VARIANTS = {"ball + IK per draw": leg_ball_ik, "IK all, then draw": leg_ik_all,
            "grid all, then draw": leg_grid_all}

def single_leg(f, n):
    return jax.jit(lambda key, body: f(key, to_shoulders(body, targets)[LEG], n))

def full_stance(f, n):
    def run(key, body):
        Xs = to_shoulders(body, targets)                           # (L, M, 3)
        idx, valid, th, foot = jax.vmap(lambda k, x: f(k, x, n))(jax.random.split(key, L), Xs)
        return idx.T, valid.all(0), jnp.swapaxes(th, 0, 1), jnp.swapaxes(foot, 0, 1)
    return jax.jit(run)

LEG = 0
NS = (1, 1024, 16384, 262144)

## 2. Single leg

In [ ]:
def bench(make, label):
    print(f"{label}: per sample (us) / valid share / per valid sample (us)")
    print(f"{'n':>8} | " + " | ".join(f"{v:>30}" for v in VARIANTS))
    out = {}
    for n in NS:
        cells = []
        for name, f in VARIANTS.items():
            _, call, res = timed(make(f, n), jax.random.PRNGKey(4), body0, repeats=3)
            v = float(np.mean(np.asarray(res[1])))
            out[name, n] = (call, res)
            cells.append(f"{1e6 * call / n:9.4f} / {v:4.2f} / {1e6 * call / max(v * n, 1):9.4f}")
        print(f"{n:>8} | " + " | ".join(f"{c:>30}" for c in cells))
    return out

single = bench(single_leg, f"single leg {LEG}")
_, (idx, valid, th, foot) = single["grid all, then draw", NS[-1]]
Xs0 = np.asarray(to_shoulders(body0, targets)[LEG])
err = np.linalg.norm(np.asarray(foot) - Xs0[np.asarray(idx)], axis=-1)[np.asarray(valid)]
print(f"grid foot error: median {np.median(err) * 1e3:.2f} mm, max {err.max() * 1e3:.2f} mm")

## 3. Full stance

In [ ]:
stance_res = bench(full_stance, "full stance (4 legs)")
_, (idx, valid, th, foot) = stance_res["grid all, then draw", NS[-1]]
XsA = np.asarray(to_shoulders(body0, targets))                       # (L, M, 3)
want = XsA[np.arange(L)[None], np.asarray(idx)]                      # (n, L, 3)
err = np.linalg.norm(np.asarray(foot) - want, axis=-1)[np.asarray(valid)]
print(f"grid foot error: median {np.median(err) * 1e3:.2f} mm, max {err.max() * 1e3:.2f} mm")

## 4. Body pose and footholds jointly

Every sample its own body pose, drawn in a region around a centre pose (both arguments; here
`body0` with x, y +-6 cm, z +-3 cm, roll / pitch +-5 deg, yaw +-10 deg), so no per-pose scan of all
footholds. Instead, once: per leg, the footholds
within R + delta of its nominal shoulder (delta: the largest shoulder displacement over the
region), a superset of what any body in the region can reach. Note: delta is the max over
sampled bodies, slightly below the true max (~103 vs. ~111 mm bound), so footholds reachable
only from the most extreme bodies can be missed; accepted. Per sample and leg: `TRIES`
footholds from that superset, IK or lookup on each, the first reachable one (and a random
valid branch). Valid: every sampled leg found one.

In [ ]:
TRIES = 8

def draw_body(key, center, lo, hi):
    # A body pose in the region: offset (x, y, z, m) and roll / pitch / yaw (deg) in [lo, hi],
    # in the centre pose's frame.
    u = jax.random.uniform(key, (6,), minval=lo, maxval=hi)
    return center @ SE3.from_te(u[:3], jnp.deg2rad(u[3:]), "xyz")

def superset(key, center, lo, hi, n_delta):
    # Per leg, the footholds within R + delta of its shoulder at the centre pose, as a
    # cumulative count over all footholds (L, M) for drawing; and delta.
    # delta = max shoulder displacement |s(body) - s0| over the region. Any foothold x
    # reachable from a sampled body has |x - s| <= R, so |x - s0| <= R + delta (triangle
    # inequality): the R + delta ball around s0 holds them all.
    # NOTE: delta is the max over n_delta sampled bodies, which slightly underestimates the true
    # max (samples rarely hit the region's corners; for the default region ~103 mm sampled vs.
    # an upper bound |t|_max + 2 r sin(phi_max / 2) ~ 90 + 21 = 111 mm). Footholds reachable only
    # from the most extreme bodies can be missed. Accepted.
    s0 = robot.shoulders(center).translation()                               # (L, 3)
    shs = jax.vmap(lambda k: robot.shoulders(draw_body(k, center, lo, hi)).translation())(
        jax.random.split(key, n_delta))
    delta = jnp.linalg.norm(shs - s0[None], axis=-1).max()
    mask = jnp.linalg.norm(targets[None] - s0[:, None], axis=-1) <= R + delta
    return jnp.cumsum(mask, -1), delta

def leg_tries(key, sh, csum, method):
    # One leg, all n samples: TRIES footholds each from the superset, the first reachable one
    # and a random valid branch. sh: (n,) shoulder poses; csum: (M,) the leg's superset counts.
    k1, k2 = jax.random.split(key)
    n_ = sh.wxyz_xyz.shape[0]
    u = jax.random.uniform(k1, (n_, TRIES)) * csum[-1]
    f = jnp.minimum(jnp.searchsorted(csum, u, side="right"), M - 1)        # (n, TRIES), inverse CDF
    Xs = jax.vmap(lambda s, x: jax.vmap(s.inverse().apply)(x))(sh, targets[f])
    if method == "grid":
        _, ok, th, feet = jax.vmap(jax.vmap(g.lookup))(Xs)
    else:
        ok, th = jax.vmap(jax.vmap(reach_exact))(Xs)
        feet = jnp.broadcast_to(Xs[..., None, :], th.shape[:-1] + (3,))
    r = jnp.arange(n_)
    t = jnp.argmax(ok.any(-1), -1)                                          # first reachable try
    ok_t = ok[r, t]                                                         # (n, B)
    b = jnp.argmax(jnp.where(ok_t, jax.random.gumbel(k2, ok_t.shape), -jnp.inf), -1)
    foot = jax.vmap(lambda s, x: s.apply(x))(sh, feet[r, t, b])             # stored foot, world
    return ok_t.any(-1), f[r, t], th[r, t, b], foot

def joint_sampler(legs, method, n, n_delta=4096):
    # Jitted (key, center, lo, hi) -> (valid, bodies, footholds, thetas, feet, delta) for n
    # samples. center and the region are traced: changing them does not recompile.
    legs = tuple(legs)
    @jax.jit
    def run(key, center, lo, hi):
        k_sup, k_body, *k_legs = jax.random.split(key, 2 + len(legs))
        csum, delta = superset(k_sup, center, lo, hi, n_delta)
        bodies = jax.vmap(lambda k: draw_body(k, center, lo, hi))(jax.random.split(k_body, n))
        sh = jax.vmap(robot.shoulders)(bodies)                              # (n, L)
        out = [leg_tries(k, sh[:, i], csum[i], method) for k, i in zip(k_legs, legs)]
        valid = jnp.stack([o[0] for o in out], 1).all(1)
        return (valid, bodies, *(jnp.stack([o[j] for o in out], 1) for j in (1, 2, 3)), delta)
    return run

LO = jnp.array([-0.06, -0.06, -0.03, -5.0, -5.0, -10.0]); HI = -LO

print(f"per sample (us) / valid share / per valid sample (us); {TRIES} tries per leg")
print(f"{'n':>7} | " + " | ".join(f"{a + ', ' + m:>34}" for a in ("single leg", "full posture") for m in ("IK", "grid")))
joint = {}
for n in (1024, 16384, 262144):
    cells = []
    for legs in ((LEG,), tuple(range(L))):
        for method in ("ik", "grid"):
            _, call, res = timed(joint_sampler(legs, method, n), jax.random.PRNGKey(6), body0, LO, HI, repeats=3)
            v = float(np.mean(np.asarray(res[0])))
            joint[len(legs), method, n] = (call, res)
            cells.append(f"{1e6 * call / n:8.4f} / {v:4.2f} / {1e6 * call / max(v * n, 1):8.4f}")
    print(f"{n:>7} | " + " | ".join(f"{c:>34}" for c in cells))

valid, _, f, _, foot, delta = joint[L, "grid", 262144][1]
err = np.linalg.norm(np.asarray(foot) - np.asarray(targets)[np.asarray(f)], axis=-1)[np.asarray(valid)]
print(f"delta {float(delta) * 1e3:.0f} mm | grid foot error (full posture): median {np.median(err) * 1e3:.2f} mm, "
      f"max {err.max() * 1e3:.2f} mm")

In [ ]:
# Another centre and region: same compiled function, no recompile.
sampler = joint_sampler(tuple(range(L)), "grid", 16384)
jax.block_until_ready(sampler(jax.random.PRNGKey(7), body0, LO, HI))
center = body0 @ SE3.from_te(jnp.array([0.1, -0.05, 0.02]), jnp.deg2rad(jnp.array([0.0, 0.0, 20.0])), "xyz")
for c, lo, hi in ((body0, LO, HI), (center, LO, HI), (center, 2 * LO, 2 * HI)):
    t0 = time.perf_counter()
    valid, *_, delta = jax.block_until_ready(sampler(jax.random.PRNGKey(8), c, lo, hi))
    print(f"{(time.perf_counter() - t0) * 1e3:6.2f} ms | delta {float(delta) * 1e3:4.0f} mm | valid {float(valid.mean()):.2f}")

Findings (2026-10-09), 8 tries per leg, centre `body0`, superset ~1800 footholds per leg
(delta ~100 mm):

| n | single leg IK | single leg grid | full posture IK | full posture grid |
|---|---|---|---|---|
| 16384 | 1.80 / 0.90 / 2.00 | 0.61 / 0.90 / 0.68 | 4.57 / 0.63 / 7.29 | 1.42 / 0.63 / 2.27 |
| 262144 | 1.45 / 0.90 / 1.61 | 0.38 / 0.90 / 0.42 | 4.16 / 0.63 / 6.58 | 1.05 / 0.63 / 1.67 |

(us per sample / valid share / us per valid sample; per call, including the superset.) Grid
~3-4x faster than IK. 1M full postures ~1 s; a single leg ~2.8x cheaper than a full posture.
Foot error median 4.9 mm, max 8.6 mm. Centre and region are traced: a new centre or region
costs no recompile (16384 full postures in ~23 ms). Doubling the region doubles delta and
lowers the valid share (0.61 -> 0.28): the superset grows, its reachable share drops.

Scoring is the bottleneck: the push score with its statics costs ~4.7 us per posture
(~120 ms for 25,600, vs. ~33 ms to sample them), ~85% of it the MuJoCo statics. Options in
`lab/gait_graph/planner/docs/push-score.md` ("Cost: the statics are the bottleneck").

An earlier version (legs vmapped inside each sample, the superset fixed for `body0`) had a
single leg cost about as much as a full posture; with the legs as an outer loop over all
samples at once, the costs separate as expected.